# Omni NCM — test C (3 modalities, modality experts, heterogeneous head)

Runs on a free Colab **T4/A100**. ~20 minutes end to end at the default settings.

What this proves before any money is spent: a model with **three modality experts** and a **split
head** (levels 0-4 = 256-way, finest text level = 8000-way) actually learns, that the preflight
gate lets it start, and that every loss lands in TensorBoard.

Video is gone: a frame is an image, so frames are image samples.

In [ ]:
# ---- env: dotenv first (local .env), then Colab secrets, then the process environment ----
import os, sys, subprocess

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "python-dotenv"], check=True)
    from dotenv import load_dotenv
load_dotenv()

def _secret(name):
    v = os.environ.get(name)
    if v and v.strip():
        return v.strip()
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v and v.strip():
            return v.strip()
    except Exception:
        pass
    return ""

GITHUB_TOKEN, HF_TOKEN = _secret("GITHUB_TOKEN"), _secret("HF_TOKEN")
assert GITHUB_TOKEN, "set GITHUB_TOKEN (Colab secret, .env, or env) - the repo is private"
os.environ["HF_TOKEN"] = HF_TOKEN
print("token length", len(GITHUB_TOKEN), "| hf token", "set" if HF_TOKEN else "MISSING")
print("python", sys.executable)

In [ ]:
# ---- clone or update the new repo ----
REPO, PKG = "adi274903/ncm-omni-v2", "/content/ncm_omni_v2"

askpass = "/tmp/_git_askpass.sh"
with open(askpass, "w") as f:                       # token never enters argv or cell output
    f.write("#!/bin/sh\ncase \"$1\" in *sername*) echo x-access-token ;; *) echo %s ;; esac\n"
            % GITHUB_TOKEN)
os.chmod(askpass, 0o700)
env = dict(os.environ, GIT_ASKPASS=askpass, GIT_TERMINAL_PROMPT="0")
url = f"https://github.com/{REPO}.git"

if not os.path.isdir(os.path.join(PKG, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", url, PKG], check=True, env=env)
else:
    subprocess.run(["git", "fetch", "origin", "main"], cwd=PKG, check=True, env=env)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=PKG, check=True, env=env)
os.chdir(PKG); sys.path.insert(0, PKG)
print(subprocess.check_output(["git", "log", "--oneline", "-1"], text=True).strip())

In [ ]:
# ---- the extra readers + tensorboard + tqdm ----
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "datasets", "soundfile", "av", "tensorboard", "tqdm"], check=True)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# ---- invariants: must print PIPELINE CHECK OK, 7/7 ----
import run_omni
run_omni.main(["check", "--preset", "md", "--base-len", "64"])

In [ ]:
# ---- TensorBoard BEFORE the train cell: losses appear here live ----
%load_ext tensorboard
%tensorboard --logdir /content/ncm_omni_v2/run/tb

In [ ]:
# ---- smoke: 6 steps that must get past the preflight gate (proves tb + val + tqdm) ----
# preflight runs automatically inside the trainer and REFUSES to start if the head does not learn.
import os, subprocess, sys, time

PKG = "/content/ncm_omni_v2"

def live(argv, head=""):
    print(f"{head}[{time.strftime('%H:%M:%S')}]", " ".join(argv), flush=True)
    t0 = time.time()
    p = subprocess.Popen([sys.executable, "-u", "run_omni.py"] + argv,
                         cwd=PKG, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    n = 0
    for line in p.stdout:
        n += 1
        print(f"{time.strftime('%H:%M:%S')} (+{time.time()-t0:5.0f}s) {line.rstrip()}", flush=True)
    p.wait()
    print(f"[{time.strftime('%H:%M:%S')}] exit {p.returncode}, {n} lines, "
          f"{(time.time()-t0)/60:.1f} min", flush=True)
    return p.returncode

rc = live(["train", "--arms", "strict", "--preset", "sm", "--base-len", "64",
           "--batch", "4", "--steps", "6", "--holdout-mult", "4", "--live",
           "--moe", "3", "--hetero-vocab", "8000",
           "--preflight-steps", "100", "--eval-every", "3",
           "--save-every", "6", "--log-every", "1"], head="SMOKE start\n")
print("smoke exit", rc, "- 0 means the gate passed, val ran and a checkpoint was written")

In [ ]:
# ---- inspect what the smoke left on disk ----
import glob, json, os
print("tb files:", [os.path.basename(f) for f in glob.glob("/content/ncm_omni_v2/run/tb/events*")])
for arm in ("strict",):
    f = f"/content/ncm_omni_v2/run/train_{arm}/final.pt"
    print(arm, "final.pt:", os.path.exists(f),
          f"({os.path.getsize(f)/1e6:.1f} MB)" if os.path.exists(f) else "")
    m = f"/content/ncm_omni_v2/run/metrics_{arm}.jsonl"
    if os.path.exists(m):
        rows = [json.loads(l) for l in open(m) if l.strip()]
        val = [r for r in rows if r.get("val")]
        print(f"  {len(rows)} metric rows, {len(val)} validation rows")
        for r in val[:4]:
            print("   ", r)


In [ ]:
# ---- the real test C: 300 steps at ~10M with the split head, then the table ----
live(["train", "--arms", "strict,ar", "--preset", "sm", "--base-len", "64",
      "--batch", "8", "--steps", "300", "--holdout-mult", "4", "--live",
      "--moe", "3", "--hetero-vocab", "8000",
      "--preflight-steps", "300", "--eval-every", "100",
      "--save-every", "100", "--log-every", "10"], head="TEST C start\n")

## Reading it

- **Pass**: preflight prints a table where every level `k >= 1` drops at least 1.5 nats and sits
  under half of `ln V`; the text 8000-way head lands far below `ln 8000 = 8.99`.
- **Kill**: the finest head stays pinned near `ln 8000` while coarse levels converge — that is a
  broken head, and no amount of tokens fixes it.
- Validation loss every 100 steps is written to `run/metrics_*.jsonl` (`"val": true`) and to
  TensorBoard as `strict/val_loss` and `strict/val_text_L5`.
- A run that exits at the gate is the gate doing its job, not a crash. `--no-preflight` exists only
  to debug the trainer itself.